In [1]:
from os import times

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tpqoa
from scipy import stats
from scipy.stats import mannwhitneyu
from scipy.stats import ks_2samp

In [92]:
api = tpqoa.tpqoa('oanda.cfg')

In [93]:
df = api.get_history('XAU_USD', '2015', '2026-04', 'H1', 'B')

In [94]:
df

,o,h,l,c,volume,complete
time,,,,,,
2015-01-01 23:00:00,1183.371,1187.389,1182.920,1186.728,642,True
2015-01-02 00:00:00,1186.681,1188.220,1186.121,1187.366,570,True
2015-01-02 01:00:00,1187.346,1187.346,1184.270,1184.965,418,True
2015-01-02 02:00:00,1185.008,1185.385,1184.206,1185.028,228,True
2015-01-02 03:00:00,1184.986,1186.105,1184.435,1186.105,223,True
...,...,...,...,...,...,...
2026-03-31 18:00:00,4654.010,4671.040,4653.110,4667.190,65043,True
2026-03-31 19:00:00,4667.190,4685.380,4661.450,4682.480,68296,True
2026-03-31 20:00:00,4682.370,4686.790,4664.330,4667.220,24448,True


In [95]:
df.to_csv('XAU_USD_1H.csv')

In [99]:
class SupremeBacktester(tpqoa.tpqoa):
    """
        This is a very comprehensive script, used to backtest across alot of strategies, it just runs and give results
        all the user does is to input a comprehensive timeseries dataset that with the following columns:o, h, l, and c
    """
    def __init__(self, data, conf_file, forward_return = 10):
        super().__init__(conf_file)
        self.data = data
        self.data['highest'] = (self.data['h'].shift(-10).rolling(10).max()-self.data['c'])/self.data['c']
        self.data['lowest'] = (self.data['l'].shift(-10).rolling(10).min()-self.data['c'])/self.data['c']
        self.forward_return = forward_return

        self.data['body'] = (self.data['c'] - self.data['o']).abs()

        self.data['upper_wick'] = self.data['h'] - self.data[['o', 'c']].max(axis=1)

        self.data['lower_wick'] = self.data[['o', 'c']].min(axis=1) - self.data['l']

        self.data['total_range'] = self.data['h'] - self.data['l']

        self.data['hour'] = self.data.index.hour

        self.data['forward_return'] = (self.data['c'].shift(-self.forward_return) - self.data['c'])/self.data['c']

        self.data['year'] = self.data.index.year

        self.data['time'] = self.data.index

        self.data['true_range'] = np.maximum.reduce([self.data['h'] - self.data['l'],
                                                          self.data['h'] - self.data['c'].shift(periods=1).abs(),
                                                          self.data['l'] - self.data['c'].shift(periods=1).abs()
                                                          ])

        self.data['direction'] = np.sign(self.data['c'] - self.data['o'])

    def bullish_ob(self, displacement_mult=2.0, forward_window=10):
        """
        data: data with ['o', 'h', 'l', 'c', 'ATR_14']
        type: either bullish or bearish
        displacement_mult: How much stronger the move must be than the OB candle to count.
        forward_window: How many candles to look ahead for return after a hit.
        """
        obs = []
        active_zones = []

        for i in range(1, len(self.data) - 1):
            curr = self.data.iloc[i]
            prev = self.data.iloc[i - 1]
            # --- 1. IDENTIFY NEW ORDER BLOCKS ---
            # Bullish OB: Last Bearish candle before a strong Bullish move
            if curr['c'] > curr['o'] and (curr['c'] - curr['o']) > (prev['h'] - prev['l']) * displacement_mult and \
                    prev['body'] < prev['ATR_14']:
                if prev['c'] < prev['o']:
                    active_zones.append({
                        'type': 'Bullish',
                        'top': prev['h'],
                        'bottom': prev['l'],
                        'created_at': i,
                        'created_time': self.data['time'].iloc[i],
                        'status': 'Active'
                    })

            for zone in active_zones:
                if zone['status'] != 'Active': continue

                # Check for INVALIDATION (Body Close through zone)
                if zone['type'] == 'Bullish' and curr['c'] < zone['bottom']:
                    zone['status'] = 'Invalidated'
                    continue
                hit = False
                if zone['type'] == 'Bullish':
                    # Low enters zone, but Close stays above bottom
                    if curr['l'] <= zone['top'] <= curr['c'] and i - zone['created_at'] > 10:
                        hit = True
                if hit:
                    # Capture the Return
                    future_idx = min(i + forward_window, len(self.data) - 1)
                    future_price = self.data.iloc[future_idx]['c']
                    ret = ((future_price - curr['c']) / curr['c']) if zone['type'] == 'Bullish' else (
                                curr['c'] - future_price)

                    obs.append({
                        'Type': zone['type'],
                        'Created_At': self.data.index[zone['created_at']],
                        'time': self.data['time'].iloc[i],
                        'vol_regime': self.data['vol_regime'].iloc[i],
                        'sessions': self.data['sessions'].iloc[i],
                        'year': self.data['year'].iloc[i],
                        'Hit_At': self.data.index[i],
                        'MFE':self.data['highest'].iloc[i],
                        'MAE':self.data['lowest'].iloc[i],
                        'Return': ret,
                        'Zone_Top': zone['top'],
                        'Zone_Bottom': zone['bottom']
                    })
                    zone['status'] = 'Mitigated'  # Mark as done

        return pd.DataFrame(obs)

    def bearish_ob(self, displacement_mult=2.0, forward_window=10):
        """
                data: data with ['o', 'h', 'l', 'c', 'ATR_14']
                type: either bullish or bearish
                displacement_mult: How much stronger the move must be than the OB candle to count.
                forward_window: How many candles to look ahead for return after a hit.
                """
        obs = []
        active_zones = []
        for i in range(1, len(self.data) - 1):
            curr = self.data.iloc[i]
            prev = self.data.iloc[i - 1]
            if curr['c'] < curr['o'] and (curr['o'] - curr['c']) > (prev['h'] - prev['l']) * displacement_mult:
                if prev['c'] > prev['o']:
                    active_zones.append({
                        'type': 'Bearish',
                        'top': prev['h'],
                        'bottom': prev['l'],
                        'created_time': self.data['time'].iloc[i],
                        'created_at': i,
                        'status': 'Active'
                    })

            # --- 2. INSPECT ACTIVE ZONES ---
            for zone in active_zones:
                if zone['status'] != 'Active': continue
                if zone['type'] == 'Bearish' and curr['c'] > zone['top']:
                    zone['status'] = 'Invalidated'
                    continue

                hit = False
                if zone['type'] == 'Bearish':  # Bearish
                    if curr['h'] >= zone['bottom'] >= curr['c'] and i - zone['created_at'] > 10:
                        hit = True

                if hit:
                    future_idx = min(i + forward_window, len(self.data) - 1)
                    future_price = self.data.iloc[future_idx]['c']
                    ret = ((future_price - curr['c']) / curr['c'])

                    obs.append({
                        'Type': zone['type'],
                        'Created_At': self.data.index[zone['created_at']],
                        'time': self.data['time'].iloc[i],
                        'vol_regime': self.data['vol_regime'].iloc[i],
                        'year': self.data['year'].iloc[i],
                        'sessions': self.data['sessions'].iloc[i],
                        'Hit_At': self.data.index[i],
                        'MAE':self.data['highest'].iloc[i],
                        'MFE':self.data['lowest'].iloc[i],
                        'Return': ret,
                        'Zone_Top': zone['top'],
                        'Zone_Bottom': zone['bottom']
                    })
                    zone['status'] = 'Mitigated'  # Mark as done

        return pd.DataFrame(obs)

    def baseline_stat(self, number):
        baseline_1bar = (
            self.data
            .dropna()
            .groupby(['sessions', 'vol_regime'])['forward_return_'+str(number)+'bar']
        )
        baseline_stats = baseline_1bar.agg(
            mean='mean',
            median='median',
            std='std',
            skew='skew'
        )
        return baseline_stats

    def outcome_table(self, structure_name, baseline_stat, structure_baseline_stat):
        outcome_table_bullish_impulse = structure_baseline_stat.join(
        baseline_stat,
        lsuffix='_'+structure_name,
        rsuffix='_base'
        )

        # Add deltas
        outcome_table_bullish_impulse['mean_shift'] = (
            outcome_table_bullish_impulse['mean_'+structure_name] - outcome_table_bullish_impulse['mean_base']
        )

        outcome_table_bullish_impulse['skew_shift'] = (
            outcome_table_bullish_impulse['skew_'+structure_name] - outcome_table_bullish_impulse['skew_base']
        )

        outcome_table_bullish_impulse['std_ratio'] = (
            outcome_table_bullish_impulse['std_'+structure_name] / outcome_table_bullish_impulse['std_base']
        )

        return outcome_table_bullish_impulse

    def distribution_plot(self, name, session, vol_regime, full_data, structure_data):
        ctx = (
            (self.data['sessions'] == session) &
            (self.data['vol_regime'] == vol_regime)
        )

        plt.figure(figsize=(8,5))

        plt.hist(
            full_data.loc[ctx, 'forward_return_3bar'],
            bins=100,
            alpha=0.5,
            label='Baseline',
            density=True
        )

        plt.hist(
            structure_data.loc[ctx, 'forward_return_3bar'],
            bins=100,
            alpha=0.5,
            label= name,
            density=True
        )

        plt.axvline(0, color='black', linestyle='--')
        plt.legend()
        plt.title('3-Bar Forward Return Distribution\n'+session+' + '+vol_regime+' Vol')
        plt.show()

    def stats(self, returns):
        print(f'{self.forward_return} candles forward return mean: {returns['Return'].mean()}')
        print(f'{self.forward_return} candles forward return median: {returns['Return'].median()}')
        print(f'{self.forward_return} candles forward return std: {returns['Return'].std()}')
        print(f'{self.forward_return} candles forward return skew: {returns['Return'].skew()}')
        print(f'average win: {returns['Return'][returns['Return'] >= 0].mean()}')
        print(f'average_loss: {returns['Return'][returns['Return'] <= 0].mean()}')
        print(f'{returns['MAE'].describe()}')
        print(f'{returns['MFE'].describe()}')

    def ttest(self, returns):
        t_stat, p_value = stats.ttest_ind(
            returns['Return'],
            self.data['forward_return'],
            equal_var=False,
            nan_policy='omit'
        )
        print(f'ttest result: {p_value}')

    def MW_test(self, returns):
        stat, p = mannwhitneyu(returns['Return'].dropna(), self.data['forward_return'].dropna(),
                               alternative='two-sided')
        print(f'MW_test result: {p}')

    def ks2_test(self, returns):
        stat, p = ks_2samp(returns['Return'].dropna(), self.data['forward_return'].dropna())
        print(f'ks2_test results: {p}')

    def bootstrap_resampling(self, returns):
        boot_means = []

        for _ in range(10000):
            sample = np.random.choice(returns['Return'].dropna(),
                                      size=len(returns['Return'].dropna()),
                                      replace=True)
            boot_means.append(np.mean(sample))

        lower = np.percentile(boot_means, 2.5)
        upper = np.percentile(boot_means, 97.5)
        print(f'bootstrap resampling result: {lower}')

    def winloss_rate(self, returns):
        returns['failed'] = returns['Return'] <= 0

        failure_rate = returns['failed'].mean()
        print(f'failure rate: {failure_rate}')

        returns['passed'] = returns['Return'] >= 0

        win_rate = returns['passed'].mean()
        print(f'win rate: {win_rate}')

    def yearly(self, column, returns):
        result = []

        for year in sorted(self.data[column].unique()):

            yearly = self.data[self.data[column] == year]
            structure_year = returns[returns[column] == year]

            structure = structure_year['Return']
            baseline = yearly['forward_return']

            if len(structure) < 10:
                continue  # skip tiny samples

            struct_mean = structure.mean()
            base_mean = baseline.mean()
            diff = struct_mean - base_mean

            result.append({
                'year': year,
                'structure_mean': struct_mean,
                'baseline_mean': base_mean,
                'mean_diff': diff,
                'sample_size': len(structure)
            })
        yearly_result = pd.DataFrame(result)
        print(yearly_result)

    def column(self, column_name, returns):
        vol_results = []

        for regime in self.data[column_name].unique():

            subset = self.data[self.data[column_name] == regime]
            subset_structure = returns[returns[column_name] == regime]

            structure = subset_structure['Return']
            baseline = subset['forward_return']

            if len(structure) < 10:
                continue

            struct_mean = structure.mean()
            base_mean = baseline.mean()
            diff = struct_mean - base_mean

            vol_results.append({
                'vol_regime': regime,
                'structure_mean': struct_mean,
                'baseline_mean': base_mean,
                'mean_diff': diff,
                'sample_size': len(structure)
            })

        result = pd.DataFrame(vol_results)
        print(result)

    def time_control(self):
        '''
            this method is used to set the hourly timeframe and which sessions each timeframe are being acted.
        :return:
        '''
        conditions = [
            self.data['hour'].between(0, 8),
            self.data['hour'].between(8, 9),
            self.data['hour'].between(9, 13),
            self.data['hour'].between(13, 17),
            self.data['hour'].between(17, 22),
            self.data['hour'].between(22, 23)
        ]

        choices = [
            'asian',
            'asian/london',
            'london',
            'london/NY',
            'NY',
            'Closing'
        ]

        self.data['sessions'] = np.select(conditions, choices, default='off')

    def volatility(self, number=20):
        '''
        this method is to add the volatility column and you can edit mean range by the number param
        :param number: the number is used to get the amounts used to  calculate the range or volatility.
        :return: nothing
        '''
        self.data[f'volatility_{number}'] = self.data['total_range'].rolling(number).mean()

    def daily_range(self, number = 20):
        '''
        this method is used to add the daily range column in a dataset and can only be used with timeframe below daily timeframe
        :input: the amount of dailyh roll up required
        :return: creates the daily high, daily low, daily range, and rolling daily range based on the number inputed
        '''
        daily = self.data.resample('D')

        self.data['daily_high'] = daily['h'].transform('max')
        self.data['daily_low'] = daily['l'].transform('min')

        self.data['daily_range'] = self.data['daily_high'] - self.data['daily_low']
        self.data[f'rolling_daily_range_{number}'] = self.data['daily_range'].rolling(number).mean()

    def ATR(self, number = 20):
        '''
        this is to get the atr of the dataframe, and user can input the numbers of data to use to get the atr
        :param number: input number of rolling true range needed, or just one number
        :return: creates a column where with ATR(number)
        '''
        for n in [number]:
            self.data[f'ATR_{n}'] = self.data['true_range'].rolling(n).mean()

    def vol_regime(self):
        self.data['atr_norm'] = self.data['ATR_14'] / self.data['ATR_14'].rolling(252).mean()

        self.data['vol_regime'] = pd.qcut(
            self.data['atr_norm'],
            q=[0, 0.33, 0.66, 1.0],
            labels=['Low', 'Medium', 'High']
        )

    def arrange_data(self):
        self.time_control()
        self.volatility()
        self.ATR(14)
        self.daily_range()
        self.vol_regime()

    def analyse_strategy(self):
        returned = self.bullish_ob()
        self.stats(returned)
        self.ttest(returned)
        self.MW_test(returned)
        self.ks2_test(returned)
        self.bootstrap_resampling(returned)
        self.yearly('year', returned)
        self.column('vol_regime', returned)
        self.winloss_rate(returned)



In [100]:
supreme = SupremeBacktester(data=df, conf_file='oanda.cfg')

In [101]:
supreme.arrange_data()

In [102]:
supreme.analyse_strategy()

10 candles forward return mean: 0.0012659135766655077
10 candles forward return median: 0.0005021306947353699
10 candles forward return std: 0.007836020654324932
10 candles forward return skew: 1.7555439984909504
average win: 0.005843366625510364
average_loss: -0.004295477978005813
count    237.000000
mean      -0.004845
std        0.004790
min       -0.030540
25%       -0.005907
50%       -0.003659
75%       -0.001635
max        0.000016
Name: MAE, dtype: float64
count    237.000000
mean       0.005660
std        0.007798
min       -0.000015
25%        0.001572
50%        0.003839
75%        0.007516
max        0.073463
Name: MFE, dtype: float64
ttest result: 0.042229070275492844
MW_test result: 0.14137319671051043
ks2_test results: 0.030964130926240174
bootstrap resampling result: 0.0003087083516972264
    year  structure_mean  baseline_mean  mean_diff  sample_size
0   2015        0.001079      -0.000164   0.001244           24
1   2016        0.000104       0.000161  -0.000057      